# rPPG - Deep HR Training (Colab / Kaggle)

Trains ProPOSNet / PhysNet / TS-CAN on UBFC with subject-independent and
cross-dataset ('uneven data') validation. **Runtime -> GPU (T4), then Run all.**

In [ ]:
# 1. Setup: clone-or-update repo + install (idempotent)
!git clone https://github.com/Zero-State-Logic/rPPG.git 2>/dev/null || echo 'repo already present'
%cd /content/rPPG
!git pull -q
!pip install -q -e .
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 2. Download UBFC subjects robustly (skips Drive-quota-blocked files, resumes present)
!python scripts/download_ubfc.py --out data/UBFC --limit 20

In [ ]:
# 3. Train ProPOSNet (our novel model), subject-independent split
#    (workers=0 by default -> avoids the OpenCV fork crash on Colab)
!python scripts/train_deep.py --dataset ubfc --data-root data/UBFC --model proposnet \
    --eval-mode subject --epochs 20 --batch-size 4

In [ ]:
# 4. Baselines for the paper table
!python scripts/train_deep.py --dataset ubfc --data-root data/UBFC --model physnet --eval-mode subject --epochs 20
!python scripts/train_deep.py --dataset ubfc --data-root data/UBFC --model tscan   --eval-mode subject --epochs 20

## Cross-dataset ('uneven data') validation

Once PURE is available (train UBFC, test the different PURE distribution):

```
!python scripts/train_deep.py --dataset ubfc --data-root data/UBFC --model proposnet \
    --eval-mode cross --cross-dataset pure --cross-root data/PURE
```